In [ ]:
import os
import sys
import pandas as pd
import numpy as np

from pathlib import Path
project_root = Path(os.path.dirname(os.getcwd()))
sys.path.append(str(project_root.joinpath('src')))

from utils.decoder_functions import decode_all, distance_all, example_decoder_weights
from utils.equivariance_functions import example_equiv_projs, equivariance_summary_all
from utils.plotting_functions import plot_classification_errors, plot_orientation_errors, \
    plot_classification_errors_acrossstim, plot_orientation_errors_acrossstim, \
    plot_classification_errors_acrossrelori, plot_orientation_errors_acrossrelori, \
    plot_classification_errors_acrossC, plot_classification_errors_acrossN, plot_orientation_errors_acrossN, \
    plot_orientation_errors_acrossrefs, \
    plot_equivariance_dim, plot_equivariance_pcorr, plot_equivariance_variance, \
    plot_distance_all, plot_equivariance_distance_all, plot_relcenter_all, plot_equivariance_relcenter_all, \
    plot_example_decoder_weights
from utils.plotting_validation import plot_ccnorm_pooled_equiv, plot_ccnorm_pooled_equiv_overlay, \
    plot_ccnorm_pooled_c1c2, plot_ccnorm_pooled_vs_dim, pooled_ccnorm_table, null_band_table
from utils.equivariance_validation import trial_caches_all, pooled_ccnorm_all, pooled_ccnorm_subset_all, \
    pooled_ccnorm_vs_dim_all
from utils.multiple_comparisons import correct_folder


In [ ]:
model_df = pd.read_csv( project_root / 'params' / 'model_params.csv', header=0)
params_df = pd.read_csv( project_root / 'params' / 'fixed_params.csv', header=0, index_col=0)

In [ ]:
#colors for areas and conv layers
color_range = ['rgba(211, 211, 211, 1.0)', 'rgba(169, 169, 169, 1.0)', 'rgba(128, 128, 128, 1.0)', 'rgba(105, 105, 105, 1.0)', 'rgba(0, 0, 0, 1.0)']
color_dict = {'pixels': f'{color_range[0]}', 'V1': f'{color_range[1]}', 'LM': f'{color_range[2]}', \
              'RL': f'{color_range[3]}', 'AL': f'{color_range[4]}',\
              'conv1': f'{color_range[0]}', 'conv2': f'{color_range[1]}', 'conv3': f'{color_range[2]}', \
              'conv4': f'{color_range[3]}', 'conv5': f'{color_range[4]}'}

#colors for regularization parameters
color_range2 = ['rgba(211, 211, 211, 1.0)', 'rgba(169, 169, 169, 1.0)', 'rgba(128, 128, 128, 1.0)', 'rgba(105, 105, 105, 1.0)', 'rgba(0, 0, 0, 1.0)']
color_dict2 = {'0.0001': f'{color_range2[0]}', '0.001': f'{color_range2[1]}', '0.01': f'{color_range2[2]}', '0.1': f'{color_range2[3]}', '1': f'{color_range2[4]}'}
color_dict3 = {'0.001': f'{color_range2[0]}', '0.01': f'{color_range2[1]}', '0.1': f'{color_range2[2]}', '1': f'{color_range2[3]}', '10': f'{color_range2[4]}'}

### decoding across stimuli and relative orientations (ED Fig 2)
texture and orientation decoding sliced by stimulus and by relative orientation;
uses the default (250-neuron) decoding caches produced in plot_figures.ipynb

In [ ]:
#ED Fig 2a top
# figure stored in project_root / 'results_extended' / 'svm_stim_0.001C_250avg.pdf'
plot_classification_errors_acrossstim(model_df, params_df, project_root)
#ED Fig 2a bottom
# figure stored in project_root / 'results_extended' / 'ori_stim_0ref_250avg.pdf'
plot_orientation_errors_acrossstim(model_df, params_df, project_root)

#ED Fig 2b top/bottom
# figures stored in project_root / 'results_extended' / 'svm1_relori_.pdf' (top), 'svm2_relori_.pdf' (bottom)
plot_classification_errors_acrossrelori(model_df, params_df, project_root, color_dict)
#ED Fig 2c top/bottom
# figures stored in project_root / 'results_extended' / 'ori1_relori.pdf' (top), 'ori2_relori.pdf' (bottom)
plot_orientation_errors_acrossrelori(model_df, params_df, project_root, color_dict)

### decoding across regularization strength and neuron number (SI Fig 4a, ED Fig 4)
takes a long time to run the first time

In [ ]:
#decode across regularization parameters
c_range = [0.0001, 0.001, 0.01, 0.1, 1]
decode_all(model_df, params_df, project_root, c_range=c_range, equivariant=False, orientation_decoding=False)

#ED Fig 3a
# figure stored in project_root / 'results_extended' / 'svm_Crange_250avg.pdf'
plot_classification_errors_acrossC(model_df, params_df, project_root, color_dict2)

In [ ]:
#decode across number of neurons
params_df_mod = params_df.copy()
n_neurons_list = list(np.arange(50, 1550, 50))
n_neurons_list.append(4000)

c_range = [0.001]
for n_neurons in n_neurons_list:
    params_df_mod.loc['n_neurons', 'value'] = n_neurons
    decode_all(model_df, params_df_mod, project_root, c_range=c_range, equivariant=False)

#ED Fig 3b top/bottom
# figures stored in project_root / 'results_extended' / 'svm1_0.001C_Navg.pdf' (top), 'svm2_0.001C_Navg.pdf' (bottom)
plot_classification_errors_acrossN(model_df, params_df, project_root, color_dict, C0=c_range[0], equivariant=False)
#ED Fig 3c top/bottom
# figures stored in project_root / 'results_extended' / 'ori1_Navg.pdf' (top), 'ori2_Navg.pdf' (bottom)
plot_orientation_errors_acrossN(model_df, params_df, project_root, color_dict, equivariant=False, ymin=0)

### all neurons: robustness of the main results (SI Fig 8)
takes a long time to run the first time

In [ ]:
#all-neuron params: same pipeline as the main analysis but with the full model list and a
#larger neuron budget (n_neurons=4000) under the default additive-gaussian noise model
model_df_all = pd.read_csv( project_root / 'params' / 'model_params_all.csv', header=0)
params_df_all = pd.read_csv( project_root / 'params' / 'fixed_params_all.csv', header=0, index_col=0)

In [ ]:
### first run takes a long time
# re-runs AlexNet inference and the equivariance / decoding analyses with n_neurons=4000
# under the default additive-gaussian noise model; every cache and figure gets a '4000avg'
# token (n_neurons=4000) so the default 250-neuron results are not overwritten.
# Results are cached and reused on subsequent runs.
example_equiv_projs(model_df_all, params_df_all, project_root)
equivariance_summary_all(model_df_all, params_df_all, project_root)
decode_all(model_df_all, params_df_all, project_root, c_range=[0.001])
#single-trial caches and the pooled within-image-centered CC_norm of the equivariance model at
#N = 4000 (SI Fig 8c; no shuffle nulls)
trial_caches_all(model_df_all, params_df_all, project_root)
pooled_ccnorm_all(model_df_all, params_df_all, project_root, n_shuffle=0)


In [ ]:
#SI Fig 8: N = max (opaque, params_df_all) over N = 250 (semi-transparent, params_df), area means
#only -- overlay_params_df draws the 250-neuron layer first, the 4000-neuron layer on top
#SI Fig 8a top (Fig 1d analogue): classification decoding error
# figure stored in project_root / 'results' / 'svm_0.001C_4000avg_overlay250.pdf'
plot_classification_errors(model_df_all, params_df_all, project_root, C0=0.001, ymin=0.5, overlay_params_df=params_df)

#SI Fig 8a bottom (Fig 1e analogue): absolute orientation estimation error
# figure stored in project_root / 'results' / 'ori_0ref_4000avg_overlay250.pdf'
plot_orientation_errors(model_df_all, params_df_all, project_root, ymin=0, overlay_params_df=params_df)

#SI Fig 8b top (Fig 3b bottom analogue): dimension of the invariant and tuned subspaces
# figure stored in project_root / 'results' / 'dim_4000avg_20dim_overlay250.pdf'
plot_equivariance_dim(model_df_all, params_df_all, project_root, overlay_params_df=params_df)

#SI Fig 8b bottom (Fig 4b analogue): degrees of invariance and tuning (variance-weighted)
# figure stored in project_root / 'results' / 'pcorr_4000avg_20dim_overlay250.pdf'
plot_equivariance_pcorr(model_df_all, params_df_all, project_root, ymax=1.0, overlay_params_df=params_df)

#SI Fig 8c (ED Fig 9b analogue): pooled within-image-centered CCnorm of the equivariance model
# figure stored in project_root / 'results_extended' / 'cc_pooled_equiv_overlay_4000_250avg_centered_20dim.pdf'
plot_ccnorm_pooled_equiv_overlay(model_df_all, params_df_all, project_root, n_list=(4000, 250))

#(not in the paper) variance explained by tuned and invariant dimensions at N = 4000
# figure stored in project_root / 'results_extended' / 'var_4000avg_20dim.pdf'
plot_equivariance_variance(model_df_all, params_df_all, project_root, show_mice=False)


### response distances (ED Fig 3a bottom, ED Fig 3b)

In [ ]:
#compute pairwise response distances and relative center distances (cached after first run)
distance_all(model_df, params_df, project_root)

#ED Fig 5a bottom
# figure stored in project_root / 'results_extended' / 'distances_avg.pdf'
plot_distance_all(model_df, params_df, project_root)
# 20-dim subspace version (exploratory; not a paper figure)
# figure stored in project_root / 'results_extended' / 'distances_avg_20dim.pdf'
plot_equivariance_distance_all(model_df, params_df, project_root)

#ED Fig 5b
# figure stored in project_root / 'results_extended' / 'relcenter_avg.pdf'
plot_relcenter_all(model_df, params_df, project_root, ymax=1.3)
# 20-dim subspace version (exploratory; not a paper figure)
# figure stored in project_root / 'results_extended' / 'relcenter_avg_20dim.pdf'
plot_equivariance_relcenter_all(model_df, params_df, project_root, ymax=10)

### example decoder weights (ED Fig 5)
weights of the full-space decoders of example neurons; the paper shows mouse 2 (TX109), the figures of every row of `model_params.csv` are written

In [ ]:
#full-space decoder weights of example neurons
example_decoder_weights(model_df, params_df, project_root, C0=0.001)
# figures stored in project_root / 'results_extended' / 'example_weights' / '{model}_{layer}_example_decoder_250rand_comp_*.pdf'
plot_example_decoder_weights(model_df, params_df, project_root)

### pooled CC_norm of the equivariance model: control models, shuffle nulls, fitted dimension (ED Fig 9a-b, Fig 3b)
single-trial response caches; the pooled within-image-centered CC_norm of the equivariance model and of its
fitted per-rotation and invariant control models with 10 shuffle nulls per kind (the dashed lines of Fig 3b
bottom); and the pooled CC_norm as a function of the fitted dimension; takes a long time to run the first time
(hours per row; every row is cached and skipped afterwards)

In [ ]:
#single-trial caches, the pooled CC_norm of the equivariance and control models with the shuffle nulls
#(10 per kind, as in the paper), and the pooled CC_norm vs the fitted dimension d
trial_caches_all(model_df, params_df, project_root)
pooled_ccnorm_all(model_df, params_df, project_root, n_shuffle=10)
pooled_ccnorm_vs_dim_all(model_df, params_df, project_root, d_range=[6, 20, 50, 100, 200])

In [ ]:
#ED Fig 9b: pooled centered CC_norm, equivariance model vs the fitted per-rotation control model
# figure stored in project_root / 'results_extended' / 'cc_pooled_c1c2_centered_250avg_20dim.pdf'
plot_ccnorm_pooled_c1c2(model_df, params_df, project_root)
# statistics stored in project_root / 'results_extended' / 'ccnorm_pooled_250avg_20dim_table.csv'
ccnorm_table = pooled_ccnorm_table(model_df, params_df, project_root)

#ED Fig 9a: pooled centered CC_norm vs the fitted dimension (left: pixels + mouse areas; right: AlexNet)
# figures stored in project_root / 'results_extended' / 'ccnorm_pooled_vs_dim1_centered_250avg.pdf' and 'ccnorm_pooled_vs_dim2_centered_250avg.pdf'
plot_ccnorm_pooled_vs_dim(model_df, params_df, project_root)

#Fig 3b bottom: the shuffle nulls of the dimension counts (the dashed lines; read by plot_equivariance_dim in plot_figures.ipynb)
# table stored in project_root / 'results_extended' / 'null_bands_250avg_20dim.csv'
null_bands = null_band_table(model_df, params_df, project_root)

In [ ]:
#(not in the paper) pooled centered CC_norm of the equivariance model alone (the gain-model version is SI Fig 5f)
# figure stored in project_root / 'results_extended' / 'cc_pooled_equiv_centered_250avg_20dim.pdf'
plot_ccnorm_pooled_equiv(model_df, params_df, project_root)

In [ ]:
#(not in the paper) variance explained by tuned and invariant dimensions
# figure stored in project_root / 'results_extended' / 'var_250avg_20dim.pdf'
plot_equivariance_variance(model_df, params_df, project_root, show_mice=False)

### decoding across reference orientations (SI Fig 3)
takes a long time to run the first time

In [ ]:
#decode across different reference orientations
ref_index_list = range(15)
for ref_index in ref_index_list:
    print('Reference index: ', ref_index)
    decode_all(model_df, params_df, project_root, ref_index=ref_index, equivariant=False, class_decoding=False)
    decode_all(model_df, params_df, project_root, ref_index=ref_index, equivariant=True, class_decoding=False)

#SI Fig 3a top/bottom
# figures stored in project_root / 'results_extended' / 'ori1_refs.pdf' (top), 'ori2_refs.pdf' (bottom)
plot_orientation_errors_acrossrefs(model_df, params_df, project_root, color_dict)
#SI Fig 3b top/bottom
# figures stored in project_root / 'results_extended' / 'ori1_refs_20dim.pdf' (top), 'ori2_refs_20dim.pdf' (bottom)
plot_orientation_errors_acrossrefs(model_df, params_df, project_root, color_dict, equivariant=True)

### equivariant decoding across regularization strength (SI Fig 4b)

In [ ]:
#decode across regularization parameters
c_range = [0.001, 0.01, 0.1, 1, 10]
decode_all(model_df, params_df, project_root, c_range=c_range, equivariant=True, orientation_decoding=False)

#SI Fig 4
# figure stored in project_root / 'results_extended' / 'svm_Crange_250avg_20dim.pdf'
plot_classification_errors_acrossC(model_df, params_df, project_root, color_dict3, equivariant=True)

### scenes and textures separately (SI Fig 9)
the invariant / tuned dimensions and the pooled CC_norm of the equivariance model fitted within one image class
(leave-one-image-out within the class); takes a long time to run the first time

In [ ]:
for subset in ('scenes', 'textures'):
    equivariance_summary_all(model_df, params_df, project_root, subset=subset)
    pooled_ccnorm_subset_all(model_df, params_df, project_root, subset=subset)
    #SI Fig 9a / 9b bottom: figure stored in project_root / 'results' / f'dim_250avg_20dim_{subset}.pdf'
    plot_equivariance_dim(model_df, params_df, project_root, subset=subset)
    #SI Fig 9a / 9b top: figure stored in project_root / 'results_extended' / f'cc_pooled_equiv_centered_250avg_20dim_{subset}.pdf'
    plot_ccnorm_pooled_equiv(model_df, params_df, project_root, subset=subset)

### multiple-comparison correction of the p-values (Methods)
Holm-Bonferroni and Bonferroni corrections of the pairwise Wilcoxon tests written by the plotting functions
(families: V1>LM, V1>RL, LM>RL, LM>AL, RL>AL for the mouse areas; adjacent layers for AlexNet)

In [ ]:
#adds the columns 'p_holm' and 'p_bonf' to every *_significance*.csv of results/ and results_extended/ and
#writes the combined tables results/significance_corrected.csv and results_extended/significance_corrected.csv
significance = correct_folder(project_root)

### angular steps in the tuned subspace (SI Fig 10)
the angular position of each of the 15 rotations read directly from the fitted tuned planes (circular mean
over images, planes pooled within a neuron subsample), for pixels, the visual areas of each mouse and
AlexNet conv1-5; the step coefficient of variation xi per population is quoted in the SI. Uses the
trial-averaged response caches of the main analysis

In [ ]:
from utils.angular_steps import run_all, summarize, plot_si_panels

run_all(project_root, model_df, params_df)        # cache/angular_steps/{network}_{layer}.csv
angular_summary = summarize(project_root)         # results_extended/angular_steps_summary.csv
plot_si_panels(project_root)                      # results_extended/angular_steps_{TX108,TX109,TX111,alexnet}.pdf
angular_summary[['population', 'n_fits', 'R_mean', 'dev_mean', 'step_cv']].round(3)